In [ ]:
import os
import random
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

NUM_GRAPHS = 20        
NODES_PER_GRAPH = 15    
BATCH_SIZE = 4
EPOCHS = 5             
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED_BASE = 7
os.makedirs("models", exist_ok=True)
print(f"Using device: {DEVICE}")

def generate_synthetic_network(num_nodes=40, prob_edge=0.06, seed=None):
    if seed is None:
        random.seed(42)
        np.random.seed(42)
    else:
        random.seed(seed)
        np.random.seed(seed)

    G = nx.erdos_renyi_graph(n=num_nodes, p=prob_edge, seed=seed)

    for i in range(num_nodes):
        if i not in G:
            G.add_node(i)

    type_probs = [
        ('user', 0.45), 
        ('workstation', 0.3), 
        ('server', 0.15), 
        ('router', 0.07), 
        ('firewall', 0.03)
    ]
    types, probs = zip(*type_probs)

    for node in G.nodes:
        t = random.choices(types, probs)[0]
        vuln_prob = {
            'user': 0.05,
            'workstation': 0.12,
            'server': 0.18,
            'router': 0.10,
            'firewall': 0.02
        }

        ran = random.random()   
        threshold = vuln_prob[t]
        if ran < threshold:
            is_vul = 1
        else:
            is_vul = 0

        privilege_map = {'user': 0, 'workstation': 0, 'server': 2, 'router': 1, 'firewall': 2}
        has_firewall = 1 if t == "firewall" else 0

        G.nodes[node]['type'] = t
        G.nodes[node]['is_vulnerable'] = is_vul
        G.nodes[node]['privilege'] = privilege_map[t]
        G.nodes[node]['has_firewall'] = has_firewall

    if not nx.is_connected(G):
        comps = list(nx.connected_components(G))   
        for i in range(len(comps) - 1):
            a = random.choice(list(comps[i]))
            b = random.choice(list(comps[i + 1]))
            G.add_edge(a, b)

    return G


def mark_attack_paths(G, num_attackers=1, num_targets=1, seed=None):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    nodes = list(G.nodes())
    attack_pool = [n for n, d in G.nodes(data=True) if d["type"] in ["user", "workstation"]]
    target_pool = [n for n, d in G.nodes(data=True) if d["type"] == "server"]

    if len(target_pool) == 0:
        target_pool = nodes[-max(1, len(nodes)//8):]
    if len(attack_pool) == 0:
        attack_pool = nodes[:max(1, len(nodes)//4)]

    risky = {i: 0 for i in nodes}

    for x in attack_pool:
        for y in target_pool:
            try:
                path = nx.shortest_path(G, x, y)
                for node in path:
                    risky[node] = 1
            except:
                pass

    for n, d in G.nodes(data=True):
        if d['is_vulnerable'] == 1 and random.random() < 0.08:
            risky[n] = 1

    return risky  


def nx_to_pyg_data(G, labels_dict):
    x_list = []
    num_nodes = len(G.nodes())

    for i in range(num_nodes):
        d = G.nodes[i]
        one_hot = [0, 0, 0, 0, 0]
        if d['type'] == "user":
            one_hot = [1, 0, 0, 0, 0]
        elif d['type'] == "workstation":
            one_hot = [0, 1, 0, 0, 0]
        elif d['type'] == "server":
            one_hot = [0, 0, 1, 0, 0]
        elif d['type'] == "router":
            one_hot = [0, 0, 0, 1, 0]
        elif d['type'] == "firewall":
            one_hot = [0, 0, 0, 0, 1]

        feat = one_hot
        feat += [d['is_vulnerable'], d['privilege']/2.0, d['has_firewall']]
        x_list.append(feat)

    edges = []
    for u, v in G.edges():
        edges.append([u, v])
        edges.append([v, u])

    if len(edges) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    x = torch.tensor(x_list, dtype=torch.float)
    n = G.number_of_nodes()
    y = torch.tensor([labels_dict[i] for i in range(n)], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)


def create_synthetic_dataset(num_graphs=200, nodes_per_graph=40, seed_base=0):
    dataset = []
    for i in range(num_graphs):
        G = generate_synthetic_network(num_nodes=nodes_per_graph, prob_edge=0.06, seed=seed_base+i)
        labels = mark_attack_paths(G, num_attackers=1, num_targets=1, seed=seed_base+i)
        data = nx_to_pyg_data(G, labels)
        dataset.append(data)
    return dataset


print("Generating synthetic dataset...")
dataset = create_synthetic_dataset(NUM_GRAPHS, NODES_PER_GRAPH, SEED_BASE)
print(f"Generated {len(dataset)} graphs with {NODES_PER_GRAPH} nodes each")


class AttackGCN(torch.nn.Module):
    def __init__(self, input_features, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(input_features, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = torch.nn.Linear(hidden, 1)   

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.lin(x)
        return x.view(-1)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    ys, preds, probs = [], [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        pred = (p >= 0.5).astype(int)
        ys.append(batch.y.cpu().numpy())
        preds.append(pred)
        probs.append(p)

    y = np.concatenate(ys)
    pred = np.concatenate(preds)
    prob = np.concatenate(probs)

    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, zero_division=0)
    try:
        auc = roc_auc_score(y, prob)
    except:
        auc = float('nan')

    return {'accuracy': acc, 'f1': f1, 'auc': auc}



random.shuffle(dataset)
split = int(0.8 * len(dataset))
train_ds = dataset[:split]
test_ds = dataset[split:]

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train graphs: {len(train_ds)}, Test graphs: {len(test_ds)}")

in_channels = train_ds[0].num_node_features
model = AttackGCN(input_features=in_channels, hidden=64).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCEWithLogitsLoss()

best_f1 = -1.0
best_state = None

for epoch in range(1, EPOCHS+1):
    loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    metrics = evaluate(model, test_loader, DEVICE)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Acc: {metrics['accuracy']:.3f} | "
          f"F1: {metrics['f1']:.3f} | AUC: {metrics['auc']:.3f}")
    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_state = model.state_dict()

if best_state is not None:
    torch.save(best_state, "models/best_attack_gcn.pt")
    print(" Model saved at models/best_attack_gcn.pt")


Using device: cpu
Generating synthetic dataset...
Generated 20 graphs with 15 nodes each
Train graphs: 16, Test graphs: 4


C:\Users\psash\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 01 | Loss: 0.5761 | Acc: 0.933 | F1: 0.966 | AUC: 0.875
Epoch 02 | Loss: 0.2840 | Acc: 0.933 | F1: 0.966 | AUC: 0.902
Epoch 03 | Loss: 0.1464 | Acc: 0.933 | F1: 0.966 | AUC: 0.960
Epoch 04 | Loss: 0.1864 | Acc: 0.933 | F1: 0.966 | AUC: 0.982
Epoch 05 | Loss: 0.1539 | Acc: 0.933 | F1: 0.966 | AUC: 0.987
✅ Model saved at models/best_attack_gcn.pt
